In [1]:
import warnings
warnings.filterwarnings('ignore')
import json
import subprocess
import sys
import time
from pathlib import Path
from synthesis.llm_client import LLMClient
from configs.config_loader import Config
from synthesis.data_utils import (
    save_generated_file, load_json, extract_code_block,
    verify_python_syntax, verify_json_structure,
    parse_ref_fields, topological_sort, validate_foreign_keys,
    calculate_coverage_statistics, format_validation_report,
    sample_db,
)
notebook_config = Config("configs/app_notebook.yaml")


DOMAIN = "work"
llm_model_name = "claude-opus-4-6"
llm = LLMClient(llm_model_name, new_config=notebook_config)

# 路径配置：生成代码和数据分离存放
PROMPTS_DIR       = Path("prompts") / DOMAIN
GENERATED_DATA_DIR = Path("bench") / "data_dir/domains" / DOMAIN
GENERATED_CODE_DIR = Path("bench") / "domains" / DOMAIN
BASE_DATA_DIR      = Path("bench") / "data_dir/domains" / "base"
BASE_CODE_DIR      = Path("bench") / "domains" / "base"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)

print("✓ 初始化完成")
print(f"LLM: {llm.llm.model_name}")
print(f"Prompts 目录: {PROMPTS_DIR}")
print(f"生成数据目录：{GENERATED_DATA_DIR}")
print(f"生成代码目录：{GENERATED_CODE_DIR}")
print(f"BASE 数据目录：{BASE_DATA_DIR}")

✓ 初始化完成
LLM: claude-opus-4-6
Prompts 目录: prompts/work
生成数据目录：bench/data_dir/domains/work
生成代码目录：bench/domains/work
BASE 数据目录：bench/data_dir/domains/base


In [2]:
# 加载 BASE 域生成结果（作为 WORK 域的基础上下文）
with open(BASE_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    base_schema = f.read()
with open(BASE_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    base_models = f.read()
with open(BASE_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_data_raw = f.read()
with open(BASE_CODE_DIR / "gen_data.py", 'r', encoding='utf-8') as f:
    base_gen_data_code = f.read()

base_data = json.loads(base_data_raw)
print("加载完成")
print(f"BASE 表数量：{len(base_data)}")

加载完成
BASE 表数量：16


### 1. 生成 WORK Schema

In [4]:
with open(PROMPTS_DIR / "1_data_schema.txt", 'r', encoding='utf-8') as f:
    schema_prompt = f.read()

schema_prompt = schema_prompt.replace("{base_schema}", base_schema)

print(f"正在生成 {DOMAIN} Schema...")
print("-" * 60)
schema_response = ""
for chunk in llm.llm.stream(schema_prompt):
    content = chunk.content
    schema_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

schema_text = extract_code_block(schema_response)
is_json, schema_data = verify_json_structure(schema_text)
if is_json:
    print(f"✓ JSON 结构验证通过，共 {len(schema_data)} 张表：")
    for t in schema_data.keys():
        print(f"  - {t}")
else:
    print(f"✗ JSON 验证失败：{schema_data}")

schema_path = GENERATED_DATA_DIR / "schema.json"
save_generated_file(str(schema_path), schema_text)
print(f"\n✓ Schema 已保存到: {schema_path}")

正在生成 work Schema...
------------------------------------------------------------


我来分析这些协同办公模块，抽象核心对象，然后设计Schema。

**核心对象分析：**

1. **工作模式/办公安排** - 居家办公、灵活办公 → `work_arrangement`
2. **团建活动** → `team_activity`
3. **项目管理** - 项目计划、进度跟踪 → `project` + `project_milestone` + `project_task`
4. **日程/待办** → `schedule_event` + `todo_item`
5. **周月报/OKR** → `work_report` + `okr_objective` + `okr_key_result`
6. **文档管理** - 项目文档、产品文档、个人文档 → `document`
7. **产品需求(PRD)** → `product_requirement`
8. **软件产品目录** → `software_product`
9. **远程会议** → `meeting`
10. **认证与考试** → `certification` + `employee_certification`
11. **管理者快照** → `manager_snapshot`
12. **提醒** → 复用 `notification` + `reminder_rule`

```json
{
    "work_arrangement": {
        "WARR001": {
            "work_arrangement_ref": "WARR001",
            "employee_ref": "EMP001",
            "arrangement_type": "remote",
            "arrangement_status": "active",
            "start_date": "2025-01-15",
            "end_date": null,
            "recur

### 2. 生成 WORK Data Models

In [5]:
with open(GENERATED_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    domain_schema = f.read()

with open(PROMPTS_DIR / "2_data_models.txt", 'r', encoding='utf-8') as f:
    models_prompt = f.read()

models_prompt = models_prompt.replace("{base_models}", base_models)
models_prompt = models_prompt.replace("{domain_schema}", domain_schema)

print(f"正在生成 {DOMAIN} Data Models...")
print("-" * 60)
models_response = ""
for chunk in llm.llm.stream(models_prompt):
    content = chunk.content
    models_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

models_code = extract_code_block(models_response)
is_valid, err = verify_python_syntax(models_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {err}")

models_path = GENERATED_CODE_DIR / "models.py"
save_generated_file(str(models_path), models_code)
print(f"✓ Models 已保存到: {models_path}")

正在生成 work Data Models...
------------------------------------------------------------


```python
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field

from bench.environment.db import DB
from generated.domains.BASE.models import OfficeDB


# --- Enums as Literal types ---

ArrangementType = Literal["remote", "hybrid", "onsite", "flexible"]
ArrangementStatus = Literal["active", "inactive", "pending", "expired"]
WorkDay = Literal["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

ActivityType = Literal["team_building", "training", "workshop", "celebration", "offsite", "hackathon", "other"]
ActivityStatus = Literal["draft", "planned", "ongoing", "completed", "cancelled"]
ParticipantRole = Literal["organizer", "participant", "facilitator", "observer"]
RsvpStatus = Literal["pending", "accepted", "declined", "tentative"]

ProjectType = Literal["development", "research", "infrastructure", "maintenance", "consulting", "other"

### 3. Schema/Model一致性检查

In [6]:
validation_prompt = f"""Schema:
{domain_schema}

Data model:
{models_code}

请检查以下三点，如果都满足则回答 "yes"，否则回答 "no" 并说明原因：
1. Data model 中的字段和类型与 Schema 定义一致
2. Data model 中的字段和类型在不同位置保持一致
3. 如果 Data model 更加合理，可以视为 yes

如果是 no，给出具体修改意见。
"""

print("正在验证 Schema 和 Model 的一致性...")
validation_response = ""
for chunk in llm.llm.stream(validation_prompt):
    validation_response += chunk.content
    print(chunk.content, end="", flush=True)
print()

正在验证 Schema 和 Model 的一致性...
我来逐一检查这三点。

## 检查过程

### 1. Data model 中的字段和类型与 Schema 定义一致

逐表对比：

- **work_arrangement**: ✅ 一致
- **team_activity**: ✅ 一致
- **team_activity_participant**: ✅ 一致
- **project**: ✅ 一致
- **project_member**: ✅ 一致
- **project_milestone**: ✅ 一致
- **project_task**: ✅ 一致
- **project_risk**: ✅ 一致
- **schedule_event**: ✅ 一致
- **schedule_event_attendee**: ✅ 一致
- **todo_item**: ✅ 一致
- **reminder_rule**: ✅ 一致
- **meeting**: ✅ 一致
- **meeting_participant**: ✅ 一致
- **okr_objective**: ✅ 一致
- **okr_key_result**: ✅ 一致
- **work_report**: ✅ 一致
- **document**: ✅ 一致
- **document_collaborator**: ✅ 一致
- **product_requirement**: ✅ 一致
- **software_product**: ✅ 一致
- **certification**: Schema 中 `related_job_family` 的值是 `"Engineering"`（字符串），而 Data model 中类型标注为 `JobFamily`。但 `JobFamily` 这个类型在当前文件中**没有定义**。它可能来自 `OfficeDB` 的父模块，如果确实在父模块中定义了则可以接受。但这是一个潜在问题。
- **employee_certification**: ✅ 一致
- **exam_schedule**: ✅ 一致
- **exam_registration**: ✅ 一致
- **manager_snapshot**: ✅ 一致
- **company_info

### 4. 生成数据脚本（Faker，主流程）

In [9]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

# 分析 IT 域的外键依赖（不含 BASE 表），推导生成顺序
ref_map = parse_ref_fields(model_code)
all_tables = list(ref_map.keys())
table_order = topological_sort(all_tables, ref_map)
dependency_info = {
    table: [(field, target) for field, target in refs]
    for table, refs in ref_map.items()
}

print(f"IT 域表生成顺序（拓扑排序，共 {len(table_order)} 张表）：")
for i, t in enumerate(table_order, 1):
    print(f"  {i:2d}. {t}")
print(f"\nFK 依赖映射：")
for t, deps in dependency_info.items():
    if deps:
        print(f"  {t}: {deps}")

with open(PROMPTS_DIR / "3-1_data_script.txt", 'r', encoding='utf-8') as f:
    script_prompt = f.read()

script_prompt = script_prompt.replace("{base_data}", base_data_raw)
script_prompt = script_prompt.replace("{base_gen_data}", base_gen_data_code)
script_prompt = script_prompt.replace("{model_code}", model_code)
script_prompt = script_prompt.replace("{table_order}", str(table_order))
script_prompt = script_prompt.replace("{dependency_info}", str(dependency_info))

local_llm = LLMClient("gpt-5.4-0305-global", 128000, new_config=notebook_config)
print(f"\n正在生成 {DOMAIN} 数据脚本...")
print("-" * 60)
full_response = ""
for chunk in local_llm.llm.stream(script_prompt):
    content = chunk.content
    full_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

script_code = extract_code_block(full_response)
ok, err = verify_python_syntax(script_code)
if not ok:
    print(f"❌ 语法错误：{err}")
else:
    print("✓ Python 语法验证通过")

script_path = GENERATED_CODE_DIR / "gen_data.py"
save_generated_file(str(script_path), script_code)
print(f"✓ 数据脚本已保存到: {script_path}")

IT 域表生成顺序（拓扑排序，共 27 张表）：
   1. workarrangements
   2. teamactivities
   3. teamactivityparticipants
   4. projects
   5. scheduleeventattendees
   6. todoitems
   7. reminderrules
   8. meetings
   9. okrobjectives
  10. okrkeyresults
  11. workreports
  12. documents
  13. softwareproducts
  14. certifications
  15. examregistrations
  16. managersnapshots
  17. companyinfos
  18. projectmembers
  19. projectmilestones
  20. projecttasks
  21. projectrisks
  22. scheduleevents
  23. meetingparticipants
  24. documentcollaborators
  25. productrequirements
  26. employeecertifications
  27. examschedules

FK 依赖映射：
  workarrangements: [('work_arrangement_ref', 'work_arrangements'), ('employee_ref', 'employees'), ('approved_by_ref', 'approved_bies')]
  teamactivities: [('team_activity_ref', 'team_activities'), ('organizer_employee_ref', 'organizer_employees'), ('department_ref', 'departments'), ('team_ref', 'teams')]
  teamactivityparticipants: [('team_activity_participant_ref', 'team_ac

### 3.5 执行数据脚本，生成 db.json

In [12]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

script_path = GENERATED_CODE_DIR / "gen_data.py"
ref_map = parse_ref_fields(model_code)

print("正在执行数据生成脚本...")
result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True, timeout=120,
    cwd=str(Path(".").resolve())
)

if result.returncode != 0:
    print(f"❌ 脚本执行失败：\n{result.stderr[-3000:]}")
else:
    print("✓ 脚本执行成功")
    is_json, db_data = verify_json_structure(result.stdout)
    if not is_json:
        print(f"❌ 输出 JSON 解析失败：{db_data}")
    else:
        print(f"✓ JSON 解析成功，共 {len(db_data)} 张表")
        for tname, tdata in db_data.items():
            if isinstance(tdata, dict):
                print(f"  - {tname}: {len(tdata)} 条")

        # FK 完整性验证（WORK 域 + BASE 域合并验证）
        merged_data = {**base_data, **db_data}
        fk_ok, fk_errors = validate_foreign_keys(merged_data, ref_map)
        if fk_ok:
            print("✓ 外键完整性验证通过")
        else:
            print(f"⚠️  外键错误 {len(fk_errors)} 处（前5条）：")
            for e in fk_errors[:5]:
                print(f"  - {e}")

        db_path = GENERATED_DATA_DIR / "db.json"
        save_generated_file(str(db_path), result.stdout)
        print(f"\n✓ WORK Database 已保存到: {db_path}")

正在执行数据生成脚本...
✓ 脚本执行成功
✓ JSON 解析成功，共 27 张表
  - work_arrangement: 150 条
  - team_activity: 150 条
  - team_activity_participant: 1157 条
  - project: 150 条
  - project_member: 975 条
  - project_milestone: 691 条
  - project_task: 1472 条
  - project_risk: 381 条
  - schedule_event: 150 条
  - schedule_event_attendee: 586 条
  - todo_item: 150 条
  - reminder_rule: 150 条
  - meeting: 150 条
  - meeting_participant: 901 条
  - okr_objective: 150 条
  - okr_key_result: 442 条
  - work_report: 150 条
  - document: 150 条
  - document_collaborator: 751 条
  - product_requirement: 150 条
  - software_product: 12 条
  - certification: 12 条
  - employee_certification: 150 条
  - exam_schedule: 12 条
  - exam_registration: 84 条
  - manager_snapshot: 24 条
  - company_info: 10 条
✓ 外键完整性验证通过

✓ WORK Database 已保存到: bench/data_dir/domains/work/db.json


In [15]:
# Pydantic 模型验证：合并 BASE + WORK 数据后做完整校验
import sys as _sys
_sys.path.insert(0, str(Path(".").resolve()))
from bench.domains.work.models import WorkDB

with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    db_raw = json.load(f)

merged_raw = {**base_data, **db_raw}

try:
    work_db = WorkDB.model_validate(merged_raw)
    print("✓ WorkDB 模型验证通过")
    print(work_db.get_statistics())
except Exception as e:
    print(f"❌ 模型验证失败：{e}")

✓ WorkDB 模型验证通过
{'num_users': 75, 'num_employees': 75, 'num_departments': 12, 'num_teams': 22, 'num_roles': 8, 'num_permissions': 29, 'num_work_arrangements': 150, 'num_team_activities': 150, 'num_team_activity_participants': 1157, 'num_projects': 150, 'num_project_members': 975, 'num_project_milestones': 691, 'num_project_tasks': 1472, 'num_project_risks': 381, 'num_schedule_events': 150, 'num_schedule_event_attendees': 586, 'num_todo_items': 150, 'num_reminder_rules': 150, 'num_meetings': 150, 'num_meeting_participants': 901, 'num_okr_objectives': 150, 'num_okr_key_results': 442, 'num_work_reports': 150, 'num_documents': 150, 'num_document_collaborators': 751, 'num_product_requirements': 150, 'num_software_products': 12, 'num_certifications': 12, 'num_employee_certifications': 150, 'num_exam_schedules': 12, 'num_exam_registrations': 84, 'num_manager_snapshots': 24, 'num_company_info': 10}


### 4. 生成 WORK Tools

In [16]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

with open(PROMPTS_DIR / "4_module_tools.txt", 'r', encoding='utf-8') as f:
    tools_prompt = f.read()

# WORK 域功能描述 —— 按数据库表逐条列出，LLM 必须全部实现
work_function_description = """"协同办公": [
    {
      "title": "居家办公",
      "definition": "用户在远程工作环境下需要人工智能协助处理各类办公事务，包括但不限于远程会议管理、在线协作工具使用、工作时间规划、家庭办公环境优化建议、远程沟通技巧指导以及居家办公效率提升等相关需求。"
    },
    {
      "title": "灵活办公",
      "definition": "用户希望通过人工智能助手优化工作方式和提升办公效率，包括远程协作支持、工作时间安排、多设备同步、移动办公解决方案、以及适应不同工作环境和场景的个性化办公需求。"
    },
    {
      "title": "团建外出活动",
      "definition": "用户希望人工智能协助策划和组织团队建设活动，包括活动方案设计、场地推荐、预算规划、人员安排、流程制定等，以提升团队凝聚力和工作效率。"
    },
    {
      "title": "管理者快照",
      "definition": "用户希望获取并查看系统中所有已定义主题的完整列表和状态信息，以便了解当前可用的业务场景分类、各主题的配置情况以及使用统计，从而进行主题的统一管理和优化决策。"
    },
    {
      "title": "组织架构查询",
      "definition": "用户希望了解公司内部组织结构信息，包括查询部门设置、人员配置、汇报关系、职位层级、团队成员信息、联系方式等，以便快速定位相关人员或了解组织架构体系。"
    },
    {
      "title": "人员信息查找",
      "definition": "用户希望快速获取组织内特定人员的详细信息，包括联系方式、部门归属、职位职责、汇报关系、技能专长、项目参与情况等，以便于工作协作、人员调配或业务对接。"
    },
    {
      "title": "项目计划生成",
      "definition": "用户希望人工智能协助制定、管理和跟踪项目的整体规划，包括项目目标设定、任务分解、时间安排、资源分配、里程碑设置、风险评估以及进度监控等，以确保项目按时按质完成。"
    },
    {
      "title": "项目进度跟踪",
      "definition": "用户希望实时监控和管理项目的执行状态，包括任务完成情况、里程碑达成进度、资源分配状况、风险识别预警，以及生成进度报告和时间线可视化，确保项目按计划推进并及时调整执行策略。"
    },
    {
      "title": "日程/待办汇总",
      "definition": "用户希望人工智能帮助整理、归纳和管理个人或团队的日程安排与待办事项，包括会议时间梳理、任务优先级排序、进度跟踪、提醒设置以及生成日程总结报告等，以提高工作效率和时间管理能力。"
    },
    {
      "title": "日程/待办提醒",
      "definition": "用户希望人工智能帮助管理和跟踪个人或团队的工作安排，包括创建、修改、查询日程安排，设置任务提醒，跟踪待办事项完成状态，以及基于优先级和截止时间进行智能提醒和时间规划。"
    },
    {
      "title": "周月报/目标与关键结果(目标与关键结果)生成",
      "definition": "用户希望人工智能基于工作进展、目标完成情况和关键指标，自动生成结构化的周报、月报或目标与关键结果（目标与关键结果）文档，包括工作总结、进度跟踪、成果展示、问题分析和下阶段计划等内容。"
    },
    {
      "title": "产品软件入口查询",
      "definition": "用户希望快速找到并访问特定的办公软件、系统或应用程序，包括查询软件的登录地址、访问权限、使用方法、功能介绍或相关负责人联系方式等信息。"
    },
    {
      "title": "产品软件基本介绍",
      "definition": "用户希望快速了解某个软件产品的核心功能、特性、使用场景和基本操作方法，以便评估该软件是否符合业务需求或为团队成员提供产品培训材料。"
    },
    {
      "title": "具体产品问题咨询",
      "definition": "用户希望获得关于公司内部产品功能、使用方法、故障排除或配置设置等方面的专业指导和解答，包括软件操作步骤、系统集成问题、功能限制说明、版本更新信息等具体技术支持。"
    },
    {
      "title": "项目文档查询",
      "definition": "用户希望快速检索和获取项目相关的各类文档信息，包括项目计划书、需求文档、技术方案、进度报告、会议记录等，以便了解项目状态、获取特定信息或为决策提供支撑。"
    },
    {
      "title": "产品文档查询",
      "definition": "用户希望快速查找和获取公司内部的产品相关文档信息，包括产品需求文档、技术规格说明、用户手册、产品产品规划路线图、功能介绍、版本更新记录等，以便了解产品详情、解决问题或支持工作决策。"
    },
    {
      "title": "个人文档查询",
      "definition": "用户希望快速检索和获取个人工作相关的文档内容，包括查找历史文件、搜索文档中的特定信息、获取文档摘要、定位相关资料等，以提高个人办公效率和信息获取速度。"
    },
    {
      "title": "认证与考试",
      "definition": "用户希望获取或管理各类职业认证、技能考试相关信息，包括查询考试时间、报名流程、证书有效期、培训要求，以及跟踪团队成员的认证状态和到期提醒等。"
    },
    {
      "title": "公司基本信息查询",
      "definition": "用户希望快速获取公司的基础信息和关键数据，包括公司名称、注册地址、成立时间、法定代表人、注册资本、经营范围、股东结构、联系方式、财务状况等核心信息，以支持商务合作、尽职调查、市场分析等办公场景需求。"
    },
    {
      "title": "产品需求(产品需求)修改与调整",
      "definition": "用户希望人工智能协助处理产品需求的修改与调整工作，包括分析原始需求内容、定位需变更部分、优化需求描述、调整功能范围与优先级、补充业务规则和验收标准、评估对设计开发测试的影响，并生成更新后的需求文档或变更说明，以支持产品迭代和跨团队协同。"
    },
    {
      "title": "产品需求(产品需求)作废与关闭",
      "definition": "用户需要对项目管理中的产品需求（产品需求）进行生命周期管理，包括将不再需要的需求标记为作废状态，或正式关闭已完成的需求项目，确保项目状态的准确性和可追溯性。"
    },
    {
      "title": "远程会议设置",
      "definition": "用户希望配置和优化远程会议的各项参数和功能，包括会议室创建、参会人员邀请、会议时间安排、音视频设备调试、屏幕共享权限设置、会议录制配置等，以确保远程会议能够顺利进行并达到预期的沟通效果。"
    }
  ]
"""

with open(BASE_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    base_tools_code = f.read()

tools_prompt = tools_prompt.replace("{domain}", DOMAIN)
tools_prompt = tools_prompt.replace("{base_tools}", base_tools_code)
tools_prompt = tools_prompt.replace("{model_code}", model_code)
tools_prompt = tools_prompt.replace("{function_description}", work_function_description)

print(f"正在生成 {DOMAIN} Tools...")
print("-" * 60)
tools_response = ""
for chunk in llm.llm.stream(tools_prompt):
    content = chunk.content
    tools_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

tools_code = extract_code_block(tools_response)
is_valid, error = verify_python_syntax(tools_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {error}")

tools_path = GENERATED_CODE_DIR / "tools.py"
save_generated_file(str(tools_path), tools_code)
print(f"✓ Tools 已保存到: {tools_path}")

正在生成 work Tools...
------------------------------------------------------------


```python
"""Toolkit for the work collaboration domain, extending OfficeTools."""
from typing import Dict, List, Optional, Any
import json
from datetime import datetime, timezone

from bench.environment.toolkit import ToolType, is_tool
from bench.domains.base.tools import OfficeTools
from bench.domains.work.models import (
    WorkDB,
    WorkArrangement,
    TeamActivity,
    TeamActivityParticipant,
    Project,
    ProjectMember,
    ProjectMilestone,
    ProjectTask,
    ProjectRisk,
    ScheduleEvent,
    ScheduleEventAttendee,
    TodoItem,
    ReminderRule,
    Meeting,
    MeetingParticipant,
    OkrObjective,
    OkrKeyResult,
    WorkReport,
    Document,
    DocumentCollaborator,
    ProductRequirement,
    SoftwareProduct,
    Certification,
    EmployeeCertification,
    ExamSchedule,
    ExamRegistration,
    ManagerSnapshot,
    CompanyInfo,
)


class WorkTools(OfficeTools):
    """Toolkit 